# Digit Reader - Read the Number Written by the Student

Approach 2, step 2. For the exercises where the student writes a number (type `write_number` in the answer key, q15 to q30), a CNN reads the number written in the answer box. The verdict is then simply: is the number read equal to the expected answer?

**Steps**:
1. Cut the answer box of each copy, remove the printed box borders, and separate the digits from left to right
2. Train three CNNs to recognise one handwritten digit, on MNIST (70,000 public handwritten digits), with augmentations that make the digits look like a child's pencil
3. Read our copies with each CNN, compare them, and keep the best one

Each CNN is trained twice: on MNIST only, and on MNIST + EMNIST (6 combinations in total).

**Compared CNNs**:
- a small CNN trained from scratch (4 convolutions)
- ResNet-18, pre-trained on ImageNet, fine-tuned on MNIST
- EfficientNet-B0, pre-trained on ImageNet, fine-tuned on MNIST

**Compared training sets**:
- MNIST: 60,000 training digits, written by American Census Bureau employees and high school students
- MNIST + EMNIST: 60,000 + 240,000 training digits. EMNIST digits come from the same original NIST collection but from many more writers, so the handwriting is more varied

**Why this solves the small data problem**: the CNN never learns our exercises. It only learns to read a digit, on public data. Our 80 copies are not used for training at all, only to check and compare the readers.

**Inputs**:
- `data/aligned/qXX/<copy>.png` (output of `02_template_subtraction.ipynb`). Inside the answer box the worksheet is blank, so we read the aligned copy and not the marks (the subtraction loses part of the very pale strokes).
- `data/templates/qXX.png`: position of the printed lines
- `config/answer_zones.json`: position of the answer box of each exercise
- `config/answer_key.csv`: expected answer of each exercise

**Outputs**:
- `models/digit_reader.pth` and `models/digit_reader.json`: the best CNN, its architecture and its training set (not versioned)
- `outputs/reader/digit_reader_comparison.csv`: scores of the 6 combinations (3 CNNs × 2 training sets)
- `outputs/reader/digit_readings.csv`: the number read on each copy by the best CNN, with the expected answer and the label
- `outputs/reader/digit_reader_training.png`: training curves

## 1. Imports and configuration

In [ ]:
import json
import time
import numpy as np
import pandas as pd
import cv2
import matplotlib.pyplot as plt
from pathlib import Path
from sklearn.metrics import balanced_accuracy_score, confusion_matrix, ConfusionMatrixDisplay

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms, models

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

ALIGNED_DIR = Path("../../data/aligned")
TEMPLATES_DIR = Path("../../data/templates")
LABELED_DIR = Path("../../data/labeled/by_exercise")
MNIST_DIR = Path("../../data/external")            # MNIST and EMNIST are stored here (not versioned)
ZONES_PATH = Path("../../config/answer_zones.json")
ANSWER_KEY_PATH = Path("../../config/answer_key.csv")
MODEL_PATH = Path("../../models/digit_reader.pth")
MODEL_INFO_PATH = Path("../../models/digit_reader.json")
OUTPUT_DIR = Path("../../outputs/reader")

# Compared CNNs: epochs and learning rate of each one
CNN_SETTINGS = {
    "small_cnn": {"epochs": 8, "learning_rate": 1e-3},        # trained from scratch
    "resnet18": {"epochs": 4, "learning_rate": 5e-4},         # pre-trained, fine-tuned
    "efficientnet_b0": {"epochs": 4, "learning_rate": 5e-4},  # pre-trained, fine-tuned
}
TRAINING_SETS = ["mnist", "mnist_emnist"]                 # compared training sets
BATCH_SIZE = 256
SEED = 42

for folder in (MNIST_DIR, MODEL_PATH.parent, OUTPUT_DIR):
    folder.mkdir(parents=True, exist_ok=True)

## 2. Training digits and augmentation

**MNIST** digits are white on black, 28×28 pixels, written with a thick pen and centred. The digits of our copies are written in pencil by children: thinner or thicker strokes, tilted, sometimes very pale. During training, each digit is randomly modified so that the CNN also sees this kind of writing:
- rotation (±15°), shift, size change and shear (tilted writing)
- stroke thickness: thinner or thicker
- intensity: pale strokes (pencil pressed lightly)
- slight blur

**EMNIST** (digits split) has 240,000 training digits in the same format. Its images are stored transposed (rows and columns swapped), so they are transposed back to match MNIST.

The test set of MNIST is never augmented: it measures how well the CNN reads normal handwritten digits. It is the same for both training sets, so that they can be compared.

EMNIST is downloaded from the NIST website (https://biometrics.nist.gov/cs_links/EMNIST/gzip.zip, about 560 MB). If the automatic download fails, put the file in `data/external/EMNIST/raw/gzip.zip`: torchvision checks its MD5 checksum and extracts it.

In [ ]:
geometry = transforms.RandomAffine(degrees=15, translate=(0.1, 0.1), scale=(0.8, 1.1), shear=10)


def augment_batch(x):
    """Stroke thickness, intensity and blur changes, on a batch of digits (N,1,28,28) in [0, 1]."""
    n = x.shape[0]
    choice = torch.rand(n, device=x.device)
    thicker = F.max_pool2d(x, 3, stride=1, padding=1)
    thinner = -F.max_pool2d(-x, 2, stride=1, padding=1)[..., :28, :28]
    x = torch.where((choice < 0.25).view(-1, 1, 1, 1), thicker, x)
    x = torch.where((choice > 0.85).view(-1, 1, 1, 1), thinner, x)
    intensity = torch.empty(n, 1, 1, 1, device=x.device).uniform_(0.4, 1.0)
    x = x * intensity
    blurred = transforms.functional.gaussian_blur(x, kernel_size=3, sigma=0.8)
    x = torch.where((torch.rand(n, device=x.device) < 0.3).view(-1, 1, 1, 1), blurred, x)
    return x.clamp(0, 1)


from PIL import Image
from torch.utils.data import ConcatDataset

mnist_train = datasets.MNIST(MNIST_DIR, train=True, download=True,
                             transform=transforms.Compose([geometry, transforms.ToTensor()]))
emnist_train = datasets.EMNIST(MNIST_DIR, split="digits", train=True, download=True,
                               transform=transforms.Compose([
                                   transforms.Lambda(lambda img: img.transpose(Image.Transpose.TRANSPOSE)),  # back to the MNIST orientation
                                   geometry, transforms.ToTensor()]))
test_set = datasets.MNIST(MNIST_DIR, train=False, download=True, transform=transforms.ToTensor())

train_loaders = {
    "mnist": DataLoader(mnist_train, batch_size=BATCH_SIZE, shuffle=True, num_workers=0),
    "mnist_emnist": DataLoader(ConcatDataset([mnist_train, emnist_train]), batch_size=BATCH_SIZE, shuffle=True, num_workers=0),
}
test_loader = DataLoader(test_set, batch_size=500, shuffle=False)
print(f"MNIST: {len(mnist_train)} training digits | EMNIST: {len(emnist_train)} training digits | MNIST test: {len(test_set)} digits")

examples = augment_batch(next(iter(train_loaders["mnist_emnist"]))[0][:16].to(device)).cpu()
fig, axes = plt.subplots(1, 16, figsize=(16, 1.4))
for ax, img in zip(axes, examples):
    ax.imshow(img[0], cmap="gray")
    ax.axis("off")
plt.suptitle("Augmented training digits")
plt.show()

## 3. Cutting the digits out of the answer box

For each copy, we cut the answer zone out of the aligned image (and the same zone out of the template), then:
1. **Printed lines**: the box borders are taken from the template, at the place where they are printed, and erased from the copy. A thin band along the zone edges is also cleared
2. **Ink, adapted to each copy**: some students press hard, others write very pale. The threshold is set from the paper colour and the darkest stroke of the copy. Strong ink is kept, and weak ink only where it touches strong ink (hysteresis): a pale stroke stays in one piece without picking up the grain of the paper. A zone with no contrasted stroke is empty
3. **Pieces**: the ink is split into connected pieces. Leftovers of the box borders (pieces lying in the outer band of the zone, long flat lines) and small fragments are dropped. Pieces that overlap horizontally, or are very close, are grouped: a pencil stroke can be broken in two
4. **MNIST format**: each digit is placed in a 28×28 image like the MNIST digits: white on black, fitted in a 20×20 box keeping its proportions, centred on its centre of mass, with the stroke thickened to look like MNIST

Two touching digits (a `44` written in one go) form a single group. This is handled when reading (section 4).

The parameters were set by looking at the copies labelled correct, which are also used to measure the reading (section 6). The measured reading accuracy is therefore slightly optimistic.

In [ ]:
zones = {k: v for k, v in json.loads(ZONES_PATH.read_text()).items() if not k.startswith("_")}

PRINTED_THRESHOLD = 200     # template pixels darker than this are printed lines
PRINTED_MARGIN = 7          # the printed lines of the template are thickened by this (pixels) before being erased
STRONG_INK = 0.5            # strong ink: darker than paper - 50 % of the contrast of the darkest stroke
WEAK_INK = 0.25             # weak ink (kept only if it touches strong ink): darker than paper - 25 % of it
MIN_CONTRAST = 40           # if the darkest stroke is less contrasted than this, the zone is empty
EDGE_BAND = 0.03            # fraction of the zone cleared along its edges
MIN_PIECE_AREA = 15         # ink pieces smaller than this (pixels) are scan noise
MIN_DIGIT_HEIGHT = 0.15     # a digit is at least this fraction of the zone height
MIN_RELATIVE_HEIGHT = 0.4   # a digit is at least this fraction of the tallest digit of the zone
OUTER_BAND = 0.08           # pieces lying entirely in this outer band of the zone are box border leftovers
MERGE_GAP = 0.08            # pieces that overlap horizontally, or closer than this fraction of the digit height, are one digit
SPLIT_RATIO = 0.6           # a group wider than this relative to its height may be two touching digits
LINE_MIN_WIDTH = 0.5        # a piece wider than this fraction of the zone...
LINE_MAX_HEIGHT = 0.12      # ...and flatter than this fraction of the zone is a printed line


def ink_mask(zone, template_zone):
    """Ink of the zone (1 = ink), without the printed lines of the worksheet.

    The threshold adapts to each copy: some students press hard, others write very pale.
    Strong ink is kept, weak ink only where it touches strong ink (hysteresis), so a pale
    stroke stays connected without picking up the grain of the paper."""
    blurred = cv2.GaussianBlur(zone, (3, 3), 0).astype(np.float32)
    printed = cv2.dilate((template_zone < PRINTED_THRESHOLD).astype(np.uint8), np.ones((PRINTED_MARGIN, PRINTED_MARGIN), np.uint8))
    h, w = zone.shape
    band_y, band_x = max(1, int(h * EDGE_BAND)), max(1, int(w * EDGE_BAND))
    usable = (1 - printed).astype(bool)
    usable[:band_y], usable[h - band_y:], usable[:, :band_x], usable[:, w - band_x:] = False, False, False, False

    if not usable.any():
        return np.zeros_like(zone, np.uint8)
    paper = np.percentile(blurred[usable], 50)
    darkest = np.percentile(blurred[usable], 0.5)
    contrast = paper - darkest
    if contrast < MIN_CONTRAST:
        return np.zeros_like(zone, np.uint8)

    strong = (blurred < paper - STRONG_INK * contrast) & usable
    weak = (blurred < paper - WEAK_INK * contrast) & usable
    n, labels = cv2.connectedComponents(weak.astype(np.uint8))
    has_strong = np.zeros(n, bool)
    has_strong[np.unique(labels[strong])] = True
    has_strong[0] = False
    ink = has_strong[labels].astype(np.uint8)

    n, labels, stats, _ = cv2.connectedComponentsWithStats(ink)
    keep = np.zeros(n, bool)
    keep[1:] = stats[1:, cv2.CC_STAT_AREA] >= MIN_PIECE_AREA
    return keep[labels].astype(np.uint8)


def split_in_two(ink, box):
    """Splits a box in two, at the column with the least ink in its middle third."""
    x1, y1, x2, y2 = box
    width = x2 - x1
    profile = np.convolve(ink[y1:y2, x1:x2].sum(axis=0).astype(float), np.ones(5) / 5, mode="same")
    lo, hi = width // 3, 2 * width // 3
    cut = lo + int(np.argmin(profile[lo:hi]))
    parts = []
    for a, b in ((0, cut), (cut, width)):
        rows = np.where(ink[y1:y2, x1 + a:x1 + b].any(axis=1))[0]
        if len(rows) == 0:
            return None
        parts.append([x1 + a, y1 + rows[0], x1 + b, y1 + rows[-1] + 1])
    return parts


def split_digits(ink):
    """Bounding boxes (x1, y1, x2, y2) of the digits, from left to right."""
    h, w = ink.shape
    joined = cv2.dilate(ink, np.ones((3, 3), np.uint8))   # joins broken pencil strokes
    n, _, stats, _ = cv2.connectedComponentsWithStats(joined)
    pieces = sorted([x, y, x + bw, y + bh] for x, y, bw, bh, area in stats[1:])

    # leftovers of the box borders: pieces lying entirely in the outer band of the zone
    bx, by = OUTER_BAND * w, OUTER_BAND * h
    pieces = [p for p in pieces if not (p[2] < bx or p[0] > w - bx or p[3] < by or p[1] > h - by)]
    # leftovers of printed lines: long and flat pieces
    pieces = [p for p in pieces if not (p[2] - p[0] > LINE_MIN_WIDTH * w and p[3] - p[1] < LINE_MAX_HEIGHT * h)]
    if not pieces:
        return []

    # pieces that overlap horizontally or are very close belong to the same digit
    groups = [pieces[0]]
    for p in pieces[1:]:
        g = groups[-1]
        height = max(g[3] - g[1], p[3] - p[1])
        if p[0] - g[2] < MERGE_GAP * height:
            groups[-1] = [min(g[0], p[0]), min(g[1], p[1]), max(g[2], p[2]), max(g[3], p[3])]
        else:
            groups.append(p)

    groups = [g for g in groups if g[3] - g[1] >= MIN_DIGIT_HEIGHT * h]
    if not groups:
        return []
    # small fragments next to real digits (noise, a stray stroke) are dropped
    tallest = max(g[3] - g[1] for g in groups)
    groups = [g for g in groups if g[3] - g[1] >= MIN_RELATIVE_HEIGHT * tallest]
    return groups


def to_mnist(ink, box):
    """One digit in the MNIST format: 28x28, white on black, fitted in 20x20, centred on its centre of mass."""
    x1, y1, x2, y2 = box
    digit = ink[y1:y2, x1:x2].astype(np.uint8) * 255
    size = max(digit.shape)
    thickness = max(1, size // 10)                                   # MNIST-like stroke width
    digit = cv2.dilate(digit, np.ones((thickness, thickness), np.uint8))
    scale = 20 / max(digit.shape)
    new_w, new_h = max(1, round(digit.shape[1] * scale)), max(1, round(digit.shape[0] * scale))
    digit = cv2.resize(digit, (new_w, new_h), interpolation=cv2.INTER_AREA)

    canvas = np.zeros((28, 28), np.uint8)
    top, left = (28 - new_h) // 2, (28 - new_w) // 2
    canvas[top:top + new_h, left:left + new_w] = digit
    moments = cv2.moments(canvas)
    if moments["m00"] > 0:
        shift_x = 14 - moments["m10"] / moments["m00"]
        shift_y = 14 - moments["m01"] / moments["m00"]
        canvas = cv2.warpAffine(canvas, np.float32([[1, 0, shift_x], [0, 1, shift_y]]), (28, 28))
    return canvas

## 4. Reading the number

Each group is read by the CNN. The CNN also decides how the groups are cut into digits: a group wider than a digit can be two touching digits (`44`), or one wide digit (children often write a wide `7`). The group is read both as one digit and cut in two at the column with the least ink, and the more confident reading is kept (product of the digit probabilities).

The number is the digits put together from left to right. The confidence of the reading is that of its least sure digit. An empty box gives an empty reading.

The answer zones of all the copies are loaded once, then read by each CNN.

In [ ]:
def classify(digit_images, net):
    """Digits read by the CNN and their probabilities."""
    batch = torch.tensor(np.stack(digit_images), dtype=torch.float32).unsqueeze(1).div(255).to(device)
    with torch.no_grad():
        probs = F.softmax(net(batch), dim=1).cpu().numpy()
    return probs.argmax(1), probs.max(1)


def read_groups(ink, groups, net):
    """Reads each group. A wide group is read both as one digit and as two digits: the CNN keeps the more confident reading."""
    digits, confidences, boxes = [], [], []
    for box in groups:
        label, prob = classify([to_mnist(ink, box)], net)
        best = ([label[0]], [prob[0]], [box])
        width, height = box[2] - box[0], box[3] - box[1]
        parts = split_in_two(ink, box) if width > SPLIT_RATIO * height else None
        if parts:
            labels, probs = classify([to_mnist(ink, p) for p in parts], net)
            if probs.prod() > prob[0]:
                best = (list(labels), list(probs), parts)
        digits += best[0]
        confidences += best[1]
        boxes += best[2]
    return digits, confidences, boxes


def read_zone(ink, net):
    """Number written in the zone, confidence (of the least sure digit), digit images."""
    groups = split_digits(ink)
    if not groups:
        return "", 0.0, []
    digits, confidences, boxes = read_groups(ink, groups, net)
    return "".join(str(d) for d in digits), float(min(confidences)), [to_mnist(ink, b) for b in boxes]


# Answer zones of all the copies (ink only), loaded once
answer_key = pd.read_csv(ANSWER_KEY_PATH).set_index("exercise")
number_exercises = [q for q in answer_key.index if answer_key.loc[q, "type"] == "write_number"]
missing = [q for q in number_exercises if q not in zones]
assert not missing, f"no answer zone for {missing}"

copies = []
for exercise in number_exercises:
    x1, y1, x2, y2 = zones[exercise][0]
    template_zone = cv2.imread(str(TEMPLATES_DIR / f"{exercise}.png"), cv2.IMREAD_GRAYSCALE)[y1:y2, x1:x2]
    for path in sorted((LABELED_DIR / exercise).glob("*/*.jpg")):
        aligned = cv2.imread(str(ALIGNED_DIR / exercise / f"{path.stem}.png"), cv2.IMREAD_GRAYSCALE)
        copies.append({"copy": path.stem, "student": path.stem.rsplit("_", 1)[0], "exercise": exercise,
                       "label": path.parent.name, "expected": str(answer_key.loc[exercise, "expected"]),
                       "ink": ink_mask(aligned[y1:y2, x1:x2], template_zone)})
print(f"{len(copies)} copies loaded")


def read_all(net):
    """Reads every copy with a CNN. Returns the readings and the digit images given to the CNN."""
    net.eval()
    rows, images = [], {}
    for c in copies:
        number, confidence, digits = read_zone(c["ink"], net)
        images[c["copy"]] = digits
        rows.append({"student": c["student"], "exercise": c["exercise"], "label": c["label"], "expected": c["expected"],
                     "read": number, "confidence": round(confidence, 3),
                     "verdict": "correct" if number == c["expected"] else "incorrect"})
    return pd.DataFrame(rows), images


def score(readings):
    """Reading accuracy on the copies labelled correct, and verdict balanced accuracy on all the copies."""
    on_correct = readings[readings["label"] == "correct"]
    return {"reading_accuracy": (on_correct["read"] == on_correct["expected"]).mean(),
            "verdict_balanced_accuracy": balanced_accuracy_score(readings["label"] == "correct", readings["verdict"] == "correct")}

## 5. The three CNNs

All three take the same input: a 28×28 digit in the MNIST format, values between 0 and 1.
- **Small CNN**: two blocks of two convolutions (32 then 64 filters) with batch normalisation and max pooling, then a dense layer and the output layer (10 classes). Trained from scratch, it is made for 28×28 images.
- **ResNet-18** and **EfficientNet-B0**: pre-trained on ImageNet (colour photos, 224×224). The digit is enlarged to 64×64, copied on the 3 colour channels and normalised like ImageNet, and the last layer is replaced by a 10-class layer. All the layers are fine-tuned.

In [ ]:
class SmallCNN(nn.Module):
    def __init__(self):
        super().__init__()
        def block(c_in, c_out):
            return nn.Sequential(
                nn.Conv2d(c_in, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                nn.Conv2d(c_out, c_out, 3, padding=1), nn.BatchNorm2d(c_out), nn.ReLU(),
                nn.MaxPool2d(2),
            )
        self.features = nn.Sequential(block(1, 32), block(32, 64))   # 28 -> 14 -> 7
        self.classifier = nn.Sequential(
            nn.Flatten(), nn.Dropout(0.3), nn.Linear(64 * 7 * 7, 128), nn.ReLU(), nn.Dropout(0.3), nn.Linear(128, 10)
        )

    def forward(self, x):
        return self.classifier(self.features(x))


class PretrainedDigitNet(nn.Module):
    """An ImageNet CNN adapted to 28x28 grayscale digits."""
    def __init__(self, name):
        super().__init__()
        if name == "resnet18":
            self.net = models.resnet18(weights=models.ResNet18_Weights.IMAGENET1K_V1)
            self.net.fc = nn.Linear(self.net.fc.in_features, 10)
        elif name == "efficientnet_b0":
            self.net = models.efficientnet_b0(weights=models.EfficientNet_B0_Weights.IMAGENET1K_V1)
            self.net.classifier[-1] = nn.Linear(self.net.classifier[-1].in_features, 10)
        self.register_buffer("mean", torch.tensor([0.485, 0.456, 0.406]).view(1, 3, 1, 1))
        self.register_buffer("std", torch.tensor([0.229, 0.224, 0.225]).view(1, 3, 1, 1))

    def forward(self, x):
        x = F.interpolate(x, size=64, mode="bilinear", align_corners=False).repeat(1, 3, 1, 1)
        return self.net((x - self.mean) / self.std)


def create_cnn(name):
    return SmallCNN() if name == "small_cnn" else PretrainedDigitNet(name)


for name in CNN_SETTINGS:
    print(f"{name}: {sum(p.numel() for p in create_cnn(name).parameters()):,} parameters")

## 6. Training and comparison

Each CNN is trained on each training set with the augmentations of section 2, evaluated on the MNIST test set, then used to read our 80 copies. The number of epochs is the same for both training sets, so a training on MNIST + EMNIST sees 5 times more digits and lasts about 5 times longer.

**Scores on our copies**:
- **Reading accuracy**: on a copy labelled `correct`, the student necessarily wrote the expected answer, so the number read must be the expected answer. This measures the reader without any extra annotation.
- **Verdict balanced accuracy**: correct if the number read is the expected answer, compared with the manual labels on all the copies (0.5 = always the same answer).

The best combination is the one with the best reading accuracy on our copies (the verdict balanced accuracy breaks ties). These copies are also the ones used to measure it, so the score of the chosen CNN is slightly optimistic.

In [ ]:
def train_cnn(name, training_set):
    """Trains a CNN on a training set. Returns the trained CNN and its history."""
    train_loader = train_loaders[training_set]
    torch.manual_seed(SEED)
    np.random.seed(SEED)
    settings = CNN_SETTINGS[name]
    net = create_cnn(name).to(device)
    optimizer = torch.optim.Adam(net.parameters(), lr=settings["learning_rate"])
    scheduler = torch.optim.lr_scheduler.OneCycleLR(optimizer, max_lr=settings["learning_rate"],
                                                    epochs=settings["epochs"], steps_per_epoch=len(train_loader))
    criterion = nn.CrossEntropyLoss()
    history = {"train_loss": [], "train_acc": [], "test_acc": []}

    for epoch in range(settings["epochs"]):
        net.train()
        total_loss, correct, total = 0.0, 0, 0
        for imgs, labels in train_loader:
            imgs, labels = augment_batch(imgs.to(device)), labels.to(device)
            optimizer.zero_grad()
            out = net(imgs)
            loss = criterion(out, labels)
            loss.backward()
            optimizer.step()
            scheduler.step()
            total_loss += loss.item() * len(labels)
            correct += (out.argmax(1) == labels).sum().item()
            total += len(labels)

        net.eval()
        test_correct = 0
        with torch.no_grad():
            for imgs, labels in test_loader:
                test_correct += (net(imgs.to(device)).argmax(1).cpu() == labels).sum().item()
        history["train_loss"].append(total_loss / total)
        history["train_acc"].append(correct / total)
        history["test_acc"].append(test_correct / len(test_set))
        print(f"  {name} / {training_set} epoch {epoch + 1}/{settings['epochs']} | train loss {history['train_loss'][-1]:.4f} "
              f"acc {history['train_acc'][-1]:.4f} | MNIST test acc {history['test_acc'][-1]:.4f}")
    return net, history


results = {}
for training_set in TRAINING_SETS:
    for name in CNN_SETTINGS:
        start = time.time()
        net, history = train_cnn(name, training_set)
        readings, images = read_all(net)
        results[(name, training_set)] = {"net": net.cpu(), "history": history, "readings": readings, "images": images,
                                         "mnist_test_accuracy": history["test_acc"][-1],
                                         "minutes": (time.time() - start) / 60, **score(readings)}
        r = results[(name, training_set)]
        print(f"{name} / {training_set}: reading accuracy {r['reading_accuracy']:.3f} | "
              f"verdict balanced accuracy {r['verdict_balanced_accuracy']:.3f}\n")
        if torch.cuda.is_available():
            torch.cuda.empty_cache()

In [ ]:
comparison = pd.DataFrame({
    (name, training_set): {"MNIST test accuracy": r["mnist_test_accuracy"],
           "Reading accuracy (copies labelled correct)": r["reading_accuracy"],
           "Verdict balanced accuracy": r["verdict_balanced_accuracy"],
           "Training time (min)": r["minutes"]}
    for (name, training_set), r in results.items()
}).T.round(3)
comparison.index.names = ["CNN", "training set"]
comparison.to_csv(OUTPUT_DIR / "digit_reader_comparison.csv", encoding="utf-8-sig")
display(comparison)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for (name, training_set), r in results.items():
    style = "-" if training_set == "mnist" else "--"
    axes[0].plot(range(1, len(r["history"]["train_loss"]) + 1), r["history"]["train_loss"], style, label=f"{name} / {training_set}")
    axes[1].plot(range(1, len(r["history"]["test_acc"]) + 1), r["history"]["test_acc"], style, label=f"{name} / {training_set}")
axes[0].set_title("Training loss (augmented digits)")
axes[1].set_title("MNIST test accuracy")
for ax in axes:
    ax.set_xlabel("Epoch")
    ax.legend()
plt.tight_layout()
plt.savefig(OUTPUT_DIR / "digit_reader_training.png", dpi=150, bbox_inches="tight")
plt.show()

best_key = max(results, key=lambda k: (results[k]["reading_accuracy"], results[k]["verdict_balanced_accuracy"]))
best = results[best_key]
best_name = f"{best_key[0]} / {best_key[1]}"
torch.save(best["net"].state_dict(), MODEL_PATH)
MODEL_INFO_PATH.write_text(json.dumps({"architecture": best_key[0], "training_set": best_key[1]}, indent=1))
best["readings"].to_csv(OUTPUT_DIR / "digit_readings.csv", index=False)
best["net"].to(device)
print(f"Best CNN: {best_name}, saved to {MODEL_PATH}")

## 7. Checking the best reader on our copies

Verdict of the best CNN against the manual labels, and the copies labelled correct that it misread.

In [ ]:
readings = best["readings"]
on_correct = readings[readings["label"] == "correct"]
print(f"{best_name}: reading accuracy {best['reading_accuracy']:.3f} "
      f"({(on_correct['read'] == on_correct['expected']).sum()}/{len(on_correct)}), "
      f"verdict balanced accuracy {best['verdict_balanced_accuracy']:.3f}")

y_true = (readings["label"] == "correct").astype(int)
y_pred = (readings["verdict"] == "correct").astype(int)
ConfusionMatrixDisplay(confusion_matrix(y_true, y_pred), display_labels=["Incorrect", "Correct"]).plot(cmap="Blues", colorbar=False)
plt.title(f"{best_name} verdict vs manual label")
plt.show()

misread = on_correct[on_correct["read"] != on_correct["expected"]]
display(misread)

## 8. What the CNN sees

For the copies labelled correct that were misread: the digits as given to the CNN (MNIST format) and the reading. A wrong number of digits means the cutting failed (section 3), a right number of digits with a wrong value means the CNN misread a digit.

In [ ]:
n = len(misread)
if n:
    fig, axes = plt.subplots(n, 1, figsize=(6, 1.3 * n), squeeze=False)
    for ax, row in zip(axes[:, 0], misread.itertuples()):
        key = f"{row.student}_{row.exercise}"
        digits = best["images"][key]
        strip = np.hstack(digits + [np.zeros((28, 4), np.uint8)]) if digits else np.zeros((28, 28), np.uint8)
        ax.imshow(strip, cmap="gray")
        ax.set_title(f"{key}: read '{row.read}', expected '{row.expected}'", fontsize=9, loc="left")
        ax.axis("off")
    plt.tight_layout()
    plt.show()